# Data Cleaning & Transformation
**High Performance Data Processing — Project 1**  
Role: Data Cleaning Specialist (Member 2)  
Dataset: JobStreet Malaysia — `raw_data.json` → `cleaned_data.csv`

## Step 0: Import Libraries

In [ ]:
import json
import re
import pandas as pd

print("Pandas version:", pd.__version__)

Pandas version: 3.0.2


## Step 1: Load the Raw Data

The raw file is **JSONL format** (one JSON object per line). We read it line-by-line.

> **Colab users:** Upload `raw_data.json` first (Files panel → Upload), then set `RAW_PATH = "raw_data.json"`  
> **Jupyter local users:** Set `RAW_PATH` to the full path on your machine

In [ ]:
# ── CHANGE PATHS IF NEEDED ─────────────────────────────────
RAW_PATH    = "raw_data.json"
OUTPUT_PATH = "cleaned_data.csv"
# ────────────────────────────────────────────────────────────

records = []
parse_errors = 0

with open(RAW_PATH, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            parse_errors += 1
            print(f"  [WARN] Could not parse line {line_num}")

df = pd.DataFrame(records)

print(f"Records loaded  : {len(df):,}")
print(f"Parse errors    : {parse_errors}")
print(f"Columns         : {list(df.columns)}")
df.head(3)

Records loaded  : 105,094
Parse errors    : 0
Columns         : ['job_title', 'company', 'location', 'classification', 'salary']


,job_title,company,location,classification,salary
0,Cashier (Malaysia) 出納員 (馬來西亞),Luk Fook Prestige (Malaysia) Sdn Bhd,Kuala Lumpur,(Retail & Consumer Products),
1,EOI- Facilities Manager_Soft service (Penang),CBRE,George Town,(Real Estate & Property),
2,Senior Electrical Engineer- EHV Cables,AECOM,Damansara,(Engineering),


## Step 2: Initial Inspection

In [ ]:
print("=== Shape ===")
print(df.shape)

print("\n=== Data Types ===")
print(df.dtypes)

print("\n=== Missing / Null values ===")
print(df.isnull().sum())

print("\n=== Empty string counts ===")
for col in df.columns:
    empty = (df[col] == "").sum()
    pct   = empty / len(df) * 100
    print(f"  {col:20s}: {empty:6,}  ({pct:.1f}%)")

print("\n=== Duplicate rows ===")
print(f"  Exact duplicates: {df.duplicated().sum():,}")

=== Shape ===
(105094, 5)

=== Data Types ===
job_title         str
company           str
location          str
classification    str
salary            str
dtype: object

=== Missing / Null values ===
job_title         0
company           0
location          0
classification    0
salary            0
dtype: int64

=== Empty string counts ===
  job_title           :      8  (0.0%)
  company             :     10  (0.0%)
  location            :     13  (0.0%)
  classification      :     21  (0.0%)
  salary              : 48,138  (45.8%)

=== Duplicate rows ===
  Exact duplicates: 79,006


## Step 3: Remove Exact Duplicates

The dataset has ~79,000 duplicate rows (same job posted multiple times during crawling). Keep only the first occurrence.

In [ ]:
before = len(df)
df = df.drop_duplicates()
after  = len(df)

print(f"Rows before dedup : {before:,}")
print(f"Rows after dedup  : {after:,}")
print(f"Duplicates removed: {before - after:,}")

Rows before dedup : 105,094
Rows after dedup  : 26,088
Duplicates removed: 79,006


## Step 4: Handle Missing / Empty Values

- **job_title, company, location, classification** — required fields; rows missing any of these are dropped.
- **salary** — optional; empty salary is filled with `"Not Disclosed"`.

In [ ]:
# 4a: Strip whitespace from all string columns
str_cols = df.select_dtypes(include="object").columns
df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())

# 4b: Replace empty strings with NaN
df[str_cols] = df[str_cols].replace("", pd.NA)

# 4c: Drop rows with missing required fields
required = ["job_title", "company", "location", "classification"]
before = len(df)
df = df.dropna(subset=required)
print(f"Rows dropped (missing required fields): {before - len(df):,}")
print(f"Remaining rows: {len(df):,}")

# 4d: Fill missing salary
df["salary"] = df["salary"].fillna("Not Disclosed")

print("\nMissing values after Step 4:")
print(df.isnull().sum())

Rows dropped (missing required fields): 14
Remaining rows: 26,074

Missing values after Step 4:
job_title         0
company           0
location          0
classification    0
salary            0
dtype: int64


C:\Users\HUAWEI\AppData\Local\Temp\ipykernel_10188\1377360803.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include="object").columns


## Step 5: Clean the `classification` Column

Values like `(Accounting)` have wrapping parentheses — we strip them to get clean category names.

In [ ]:
df["classification"] = df["classification"].str.strip("()")
df["classification"] = df["classification"].str.strip()

print("Unique classifications:", df["classification"].nunique())
print("\nTop 10 categories:")
print(df["classification"].value_counts().head(10))

Unique classifications: 30

Top 10 categories:
classification
Manufacturing, Transport & Logistics      3226
Accounting                                2997
Engineering                               2836
Sales                                     2818
Information & Communication Technology    2451
Administration & Office Support           1722
Marketing & Communications                1429
Human Resources & Recruitment             1005
Retail & Consumer Products                 990
Construction                               954
Name: count, dtype: int64


## Step 6: Clean the `salary` Column

Some salary strings contain a **non-breaking space** (`\xa0`) e.g. `RM\xa06,800`. We normalise these and extract `salary_min` / `salary_max` as numeric columns.

In [ ]:
# Normalise non-breaking space
df["salary"] = df["salary"].str.replace("\xa0", " ", regex=False)
df["salary"] = df["salary"].str.strip()

def parse_salary(s):
    """Extract (min, max) numeric RM values from a salary string."""
    if s in ("Not Disclosed", ""):
        return None, None
    nums = re.findall(r"RM\s*([\d,]+)", str(s))
    if len(nums) >= 2:
        return int(nums[0].replace(",", "")), int(nums[1].replace(",", ""))
    elif len(nums) == 1:
        v = int(nums[0].replace(",", ""))
        return v, v
    return None, None

df[["salary_min", "salary_max"]] = df["salary"].apply(
    lambda s: pd.Series(parse_salary(s))
)
df["salary_min"] = pd.to_numeric(df["salary_min"], errors="coerce")
df["salary_max"] = pd.to_numeric(df["salary_max"], errors="coerce")

print(f"Rows with salary data   : {df['salary_min'].notna().sum():,}")
print(f"Rows 'Not Disclosed'    : {(df['salary'] == 'Not Disclosed').sum():,}")
print("\nSalary range (RM/month):")
print(df[["salary_min", "salary_max"]].describe())

Rows with salary data   : 12,330
Rows 'Not Disclosed'    : 13,498

Salary range (RM/month):
          salary_min     salary_max
count   12330.000000   12330.000000
mean     4241.809002    5692.658719
std      5248.710104    7178.284413
min         1.000000       1.000000
25%      2500.000000    3500.000000
50%      3500.000000    4500.000000
75%      4800.000000    6500.000000
max    225000.000000  300000.000000


## Step 7: Clean the `job_title` Column

No new column, non-ASCII rows are just dropped entirely. From your output we know that's only 493 rows out of 26,000+ so it won't affect your 100k+ record requirement significantly.


In [ ]:
df["job_title"] = df["job_title"].str.replace(r"\s+", " ", regex=True).str.strip()

# Remove rows that contain non-ASCII characters (e.g. Chinese) in job_title
df = df[~df["job_title"].apply(lambda t: bool(re.search(r"[^\x00-\x7F]", str(t))))]


print(f"Rows after removing non-ASCII titles: {len(df):,}")

Rows after removing non-ASCII titles: 25,209


## Step 8: Standardise the `location` Column

Map known abbreviations/variants to consistent names.

In [ ]:
location_map = {
    "Kl"                                   : "Kuala Lumpur",
    "KL"                                   : "Kuala Lumpur",
    "W.P. Kuala Lumpur"                    : "Kuala Lumpur",
    "Wilayah Persekutuan Kuala Lumpur"     : "Kuala Lumpur",
    "PJ"                                   : "Petaling Jaya",
    "Klang / Port Klang"                   : "Klang/Port Klang",
}
df["location"] = df["location"].replace(location_map)
df["location"] = df["location"].str.strip()

print("Top 15 locations after cleaning:")
print(df["location"].value_counts().head(15))

Top 15 locations after cleaning:
location
Kuala Lumpur                3934
Kuala Lumpur City Centre    1757
Petaling Jaya               1200
Johor Bahru                 1146
Selangor                     961
Shah Alam                    869
Penang                       766
Klang/Port Klang             567
Subang Jaya                  510
Bayan Lepas                  486
Puchong                      434
Petaling                     416
Kuching                      298
Johor                        295
Perai                        281
Name: count, dtype: int64


## Step 9: Type Validation & Final Column Order

In [ ]:
for col in ["job_title", "company", "location", "classification", "salary"]:
    df[col] = df[col].astype(str)
# salary_min / salary_max stay as float

# Logical column order
df = df[[
    "job_title", "company", "location", "classification",
    "salary", "salary_min", "salary_max"
]]

print("Final dtypes:")
print(df.dtypes)
print(f"\nFinal shape: {df.shape}")

Final dtypes:
job_title             str
company               str
location              str
classification        str
salary                str
salary_min        float64
salary_max        float64
dtype: object

Final shape: (25209, 7)


## Step 10: Final Quality Check

In [ ]:
print("=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Duplicates (should be 0) ===")
print(df.duplicated().sum())

print("\n=== Classification Distribution ===")
print(df["classification"].value_counts())

print("\n=== Preview ===")
df.head(5)

=== Missing Values ===
job_title             0
company               0
location              0
classification        0
salary                0
salary_min        13422
salary_max        13422
dtype: int64

=== Duplicates (should be 0) ===
0

=== Classification Distribution ===
classification
Manufacturing, Transport & Logistics      3145
Accounting                                2935
Engineering                               2747
Sales                                     2708
Information & Communication Technology    2375
Administration & Office Support           1657
Marketing & Communications                1402
Human Resources & Recruitment              970
Retail & Consumer Products                 947
Construction                               930
Hospitality & Tourism                      804
Healthcare & Medical                       713
Call Centre & Customer Service             654
Banking & Financial Services               593
Trades & Services                          451
Des

,job_title,company,location,classification,salary,salary_min,salary_max
1,EOI- Facilities Manager_Soft service (Penang),CBRE,George Town,Real Estate & Property,Not Disclosed,NaN,NaN
2,Senior Electrical Engineer- EHV Cables,AECOM,Damansara,Engineering,Not Disclosed,NaN,NaN
3,Management Consultant - ERP Platforms (SAP FI-CO),Accenture Malaysia,Kuala Lumpur City Centre,Information & Communication Technology,Not Disclosed,NaN,NaN
4,Accountant,Nala Groups on behalf of Nala Groups,Johor Bahru,Accounting,Not Disclosed,NaN,NaN
5,Marketing Executive,Nalagroups,Gelang Patah,Marketing & Communications,Not Disclosed,NaN,NaN


## Step 11: Export to CSV

In [ ]:
# Replace NaN in salary columns with the string "null"
#df["salary_min"] = df["salary_min"].fillna("null")
#df["salary_max"] = df["salary_max"].fillna("null")

df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
# utf-8-sig = UTF-8 with BOM so Excel opens correctly with non-ASCII characters

print(f"Saved {len(df):,} clean records to: {OUTPUT_PATH}")
print("Columns:", list(df.columns))

Saved 25,209 clean records to: cleaned_data.csv
Columns: ['job_title', 'company', 'location', 'classification', 'salary', 'salary_min', 'salary_max']


## Cleaning Summary

| Step | Operation | Details |
|------|-----------|--------|
| 1 | Load JSONL | 105,094 raw records |
| 2 | Inspect | ~79k duplicates, 48k empty salaries found |
| 3 | Remove duplicates | drop_duplicates() |
| 4 | Handle missing values | Drop rows missing required fields; salary → "Not Disclosed" |
| 5 | Clean classification | Strip parentheses `(Accounting)` → `Accounting` |
| 6 | Clean salary | Normalise `\xa0`, extract salary_min / salary_max columns |
| 7 | Clean job_title | Normalise whitespace, flag non-ASCII titles |
| 8 | Standardise location | Map abbreviations to full names |
| 9 | Type validation | Enforce correct dtypes |
| 10 | Quality check | Zero nulls in required fields, zero duplicates |
| 11 | Export | `cleaned_data.csv` (utf-8-sig for Excel compatibility) |